## Intensity prediction, Approach 1: statistics

The intention behind this approach is to set a baseline that all future ML models have to beat. 
We therefore introduce a simple statistics based lookup of the intensities.

**Concept:** The features and intensities of a fragment is can be mapped to a few
coordinates: the ion type (`b`/`y`), the fragment number, how far along the
peptide the cleavage is (relative position) and the precursor charge. 
Aggregating this information into a lookup table of the mean normalised intensities for each
n-tuple of `(ion_type, fragment_number_bucket, relative_position_bucket, charge)` a prediction
is a simple as a lookup in the table.
A Ridge regression is lateron run as a second, smoother baseline.

**Metric:** Base-peak-normalised intensities in the fixed 58-dim
layout, scored with spectral angle (Prosit standard) and
Pearson correlation

**Methology:** Grouped k-fold by *peptide sequence*, more todo

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge

import common as c

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

In [ ]:
spectra = pd.read_parquet("spectra_with_charge.parquet")
# sub sample, should be representative?
N = 80_000
spectra = spectra.sample(min(N, len(spectra)), random_state=RANDOM_STATE).reset_index(drop=True)
print("spectra used:", len(spectra))

Y = np.stack([
    c.ions_to_vector(i, v, normalize=True)
    for i, v in zip(spectra["matched_ions"], spectra["intensities_raw"])
])
M = np.stack([c.valid_slot_mask(s) for s in spectra["peptide_sequence"]])
groups = spectra["peptide_sequence"].values
charge = spectra["precursor_charge"].values
print("Y:", Y.shape, " mask coverage:", M.mean().round(3))

spectra used: 80000


Y: (80000, 58)  mask coverage: 0.436


### Mapping fragments into the coordinate system

In [3]:
def explode_fragments(idx_subset):
    rows = []
    for i in idx_subset:
        seq = spectra["peptide_sequence"].iloc[i]
        L = c.peptide_length(seq)
        z = charge[i]
        for slot in np.where(M[i])[0]:
            ion_type, number = c.slot_to_fragment(int(slot))
            rows.append((
                i, slot,
                0 if ion_type == "b" else 1,   # ion type id
                number,
                number / L,                     # relative position
                L,
                z,
                Y[i, slot],
            ))
    cols = ["spec", "slot", "ion", "number", "relpos", "L", "charge", "target"]
    return pd.DataFrame(rows, columns=cols)

### Table lookup

In [4]:
N_RELPOS_BINS = 10
MAX_NUM_BUCKET = 20

def add_keys(frag):
    frag = frag.copy()
    frag["nb"] = np.minimum(frag["number"], MAX_NUM_BUCKET)
    frag["rb"] = np.minimum((frag["relpos"] * N_RELPOS_BINS).astype(int), N_RELPOS_BINS - 1)
    return frag

def fit_lookup(frag):
    frag = add_keys(frag)
    full = frag.groupby(["ion", "nb", "rb", "charge"])["target"].mean()
    backoff = frag.groupby(["ion", "nb", "charge"])["target"].mean()
    coarse = frag.groupby(["ion", "rb"])["target"].mean()
    global_mean = frag["target"].mean()
    return full, backoff, coarse, global_mean

def predict_lookup(frag, table):
    full, backoff, coarse, global_mean = table
    frag = add_keys(frag)
    preds = np.empty(len(frag))
    full_d = full.to_dict(); back_d = backoff.to_dict(); coarse_d = coarse.to_dict()
    for j, (ion, nb, rb, z) in enumerate(
        zip(frag["ion"], frag["nb"], frag["rb"], frag["charge"])
    ):
        v = full_d.get((ion, nb, rb, z))
        if v is None:
            v = back_d.get((ion, nb, z))
        if v is None:
            v = coarse_d.get((ion, rb))
        if v is None:
            v = global_mean
        preds[j] = v
    return preds

### Ridge regression w/ hot encoding

In [ ]:
def featurize(frag):
    z = frag["charge"].values
    X = np.column_stack([
        frag["ion"].values,
        frag["number"].values,
        frag["relpos"].values,
        frag["relpos"].values ** 2,
        frag["L"].values,
        (z == 2).astype(float),
        (z == 3).astype(float),
        (z == 4).astype(float),
        (z == 5).astype(float),
        frag["ion"].values * frag["relpos"].values,
    ])
    return X

## Grouped k-fold evaluation

In [6]:
def reconstruct(frag, preds, n_spec):
    # Scatter per-fragment predictions back into dense 58-dim vectors.
    P = np.zeros((n_spec, c.VECTOR_DIM), dtype=np.float64)
    for spec, slot, val in zip(frag["spec"].values, frag["slot"].values, preds):
        P[spec, slot] = val
    return P

gkf = GroupKFold(n_splits=5)
results = {"lookup": [], "ridge": []}

for fold, (tr, te) in enumerate(gkf.split(np.arange(len(spectra)), groups=groups)):
    frag_tr = explode_fragments(tr)
    frag_te = explode_fragments(te)

    # Model A: lookup
    table = fit_lookup(frag_tr)
    pred_lu = predict_lookup(frag_te, table)

    # Model B: ridge (alpha lightly tuned below; default here)
    ridge = Ridge(alpha=1.0)
    ridge.fit(featurize(frag_tr), frag_tr["target"].values)
    pred_rg = np.clip(ridge.predict(featurize(frag_te)), 0, 1)

    P_lu = reconstruct(frag_te, pred_lu, len(spectra))
    P_rg = reconstruct(frag_te, pred_rg, len(spectra))

    res_lu = c.evaluate_vectors(Y[te], P_lu[te], M[te])
    res_rg = c.evaluate_vectors(Y[te], P_rg[te], M[te])
    results["lookup"].append(res_lu)
    results["ridge"].append(res_rg)
    print(f"fold {fold}: lookup SA={res_lu['spectral_angle']:.4f} "
          f"Pearson={res_lu['pearson']:.4f} | "
          f"ridge SA={res_rg['spectral_angle']:.4f} Pearson={res_rg['pearson']:.4f}")

fold 0: lookup SA=0.5240 Pearson=0.5692 | ridge SA=0.4310 Pearson=0.3742


fold 1: lookup SA=0.5244 Pearson=0.5690 | ridge SA=0.4308 Pearson=0.3716


fold 2: lookup SA=0.5250 Pearson=0.5711 | ridge SA=0.4318 Pearson=0.3757


fold 3: lookup SA=0.5246 Pearson=0.5694 | ridge SA=0.4318 Pearson=0.3731


fold 4: lookup SA=0.5253 Pearson=0.5714 | ridge SA=0.4316 Pearson=0.3750


In [7]:
def summarize(name, folds):
    sa = np.array([f["spectral_angle"] for f in folds])
    pe = np.array([f["pearson"] for f in folds])
    return {"model": name, "spectral_angle": sa.mean(), "sa_std": sa.std(),
            "pearson": pe.mean(), "pe_std": pe.std()}

summary = pd.DataFrame([summarize("lookup", results["lookup"]),
                        summarize("ridge", results["ridge"])])
print(summary.round(4).to_string(index=False))

 model  spectral_angle  sa_std  pearson  pe_std
lookup          0.5247  0.0004   0.5700  0.0010
 ridge          0.4314  0.0004   0.3739  0.0014


## Tune the Ridge regulariser

A small grid search over `alpha`, scored with the same grouped CV (on a reduced
number of folds for speed), demonstrates per-model optimisation.

In [8]:
alphas = [0.1, 1.0, 10.0, 100.0]
gkf2 = GroupKFold(n_splits=3)
alpha_scores = {}
splits = list(gkf2.split(np.arange(len(spectra)), groups=groups))
# Pre-explode once per fold to avoid repeated work.
exploded = [(explode_fragments(tr), explode_fragments(te), te) for tr, te in splits]

for a in alphas:
    sas = []
    for frag_tr, frag_te, te in exploded:
        r = Ridge(alpha=a).fit(featurize(frag_tr), frag_tr["target"].values)
        pred = np.clip(r.predict(featurize(frag_te)), 0, 1)
        P = reconstruct(frag_te, pred, len(spectra))
        sas.append(c.evaluate_vectors(Y[te], P[te], M[te])["spectral_angle"])
    alpha_scores[a] = np.mean(sas)
    print(f"alpha={a:6.1f}  SA={alpha_scores[a]:.4f}")

best_alpha = max(alpha_scores, key=alpha_scores.get)
print("best alpha:", best_alpha)

alpha=   0.1  SA=0.4314


alpha=   1.0  SA=0.4314


alpha=  10.0  SA=0.4314


alpha= 100.0  SA=0.4310
best alpha: 0.1


## Result and a worked example

In [9]:
best_sa = max(summary["spectral_angle"].max(), max(alpha_scores.values()))
print(f"Approach 1 best spectral angle (grouped CV): {best_sa:.4f}")

# Fit the lookup table on everything for the prediction demo.
full_frag = explode_fragments(np.arange(len(spectra)))
final_table = fit_lookup(full_frag)

def predict_intensities_approach1(peptide, precursor_charge):
    mask = c.valid_slot_mask(peptide)
    L = c.peptide_length(peptide)
    rows = []
    for slot in np.where(mask)[0]:
        ion_type, number = c.slot_to_fragment(int(slot))
        rows.append((0, slot, 0 if ion_type == "b" else 1, number,
                     number / L, L, precursor_charge, 0.0))
    frag = pd.DataFrame(rows, columns=["spec", "slot", "ion", "number",
                                       "relpos", "L", "charge", "target"])
    pred = predict_lookup(frag, final_table)
    vec = np.zeros(c.VECTOR_DIM)
    vec[frag["slot"].values] = pred
    return c.prediction_table(peptide, vec)

print(predict_intensities_approach1("LTQETNRV", 4).to_string(index=False))

Approach 1 best spectral angle (grouped CV): 0.5247


Fragment ion  Fragment Number  Intensity
           B                1   0.000733
           B                2   0.227807
           B                3   0.125054
           B                4   0.438460
           B                5   0.296038
           B                6   0.512611
           B                7   0.832574
           Y                1   0.000000
           Y                2   0.253642
           Y                3   0.100300
           Y                4   0.074600
           Y                5   0.391387
           Y                6   1.000000
           Y                7   0.591144


In [10]:
# Persist the headline score so later notebooks can compare against it.
np.save("approach1_score.npy", np.array([best_sa]))
summary.round(4)

,model,spectral_angle,sa_std,pearson,pe_std
0,lookup,0.5247,0.0004,0.5700,0.0010
1,ridge,0.4314,0.0004,0.3739,0.0014
